# Load a model and chat

This standalone tutorial loads one Hugging Face model, sends messages, and releases it. No earlier tutorial needs to be run. RAG is not used yet.


The following code cells find QAChat and check its `.venv` environment.
Missing packages are repaired with `uv`.

If asked, select the printed `.venv` kernel and rerun the cell.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"Root directory: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


In [ ]:
import platform

print(f"Computer system: {platform.system()}")
print(f"Computer machine: {platform.machine()}")


## 1. See models already cached

This reads Hugging Face's shared cache. Each printed name is a **model ID**. Copy the model ID you want into `MODEL_ID` in the next section. For example: `MODEL_ID = "Qwen/Qwen3-4B-MLX-4bit"`.

This is a list of cached model candidates, not a list of guaranteed chat models. The cache may also contain embedding, image, or incompatible models. Choose a text-generation/chat model whose weights match MLX or PyTorch, depening on your system.

In [ ]:
from qachat import available_models


models = available_models()
if models:
    print("Model IDs found in the Hugging Face cache:\n")
    for number, model in enumerate(models, start=1):
        size_gb = model["size_bytes"] / 1_000_000_000
        print(f"{number}. {model['model_id']} ({size_gb:.2f} GB)")
else:
    print("No models are cached yet.")


## 2. Choose the model and engine

Use `auto`, `mlx`, or `torch`. On Apple Silicon, `auto` chooses MLX.

In [ ]:
BACKEND = "auto"


To use a cached model, copy its exact model ID from the list above, for example, `MODEL_ID = "Qwen/Qwen3-4B-MLX-4bit"`.

`THINKING` is passed to compatible chat templates.

`TEMPERATURE = 0.0` gives repeatable greedy outputs.

`MAX_NEW_TOKENS = None` adds no QAChat length cap; generation uses the model's remaining context. Set a positive number to use less output tokens.

If `ALLOW_DOWNLOAD = True` and files are missing, Hugging Face downloads them to its normal cache during the loading cell.


In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-MLX-4bit"
THINKING = False
TEMPERATURE = 0.0
MAX_NEW_TOKENS = None
ALLOW_DOWNLOAD = True


## 3. Load the selected model

This is the only cell that loads weights. Loading can use several GB of memory.

In [ ]:
if "session" in globals():
    session.close()  # It closes an earlier session before creating a new one.


In [ ]:
from qachat import ChatSession


session = ChatSession(
    MODEL_ID,
    backend=BACKEND,
    thinking=THINKING,
    temperature=TEMPERATURE,
    max_new_tokens=MAX_NEW_TOKENS,
    system_prompt="Answer clearly and briefly.",
    allow_download=ALLOW_DOWNLOAD,
)


In [ ]:
print(f"Loaded model: {session.model_id}")
print(f"Backend: {session.backend}")
print(f"Device: {session.device}")


## 4. Chat

Change `question` as you prefer.


In [ ]:
question_1 = "Hello, I am Iok-ui! Please, greet me in one short sentence."

print(f"QUESTION 1: {question_1}")


In [ ]:
answer_1 = session.reply(question_1)

print(f"ANSWER 1: {answer_1}")


The second message receives the earlier turn as conversation history.


In [ ]:
question_2 = "Copy the name from my first message. Answer with the name only."

print(f"QUESTION 2: {question_2}")


In [ ]:
answer_2 = session.reply(question_2)

print(f"ANSWER 2: {answer_2}")


## 5. Release the model

Closing releases this session's model memory and clears its messages. Cached model files remain available for later use.

In [ ]:
session.close()

print("Chat closed.")
